# Notebook 06b: Feature Importance Expanded

**Purpose:** Multi-method feature importance analysis with SHAP, permutation, and linear methods

**Key Tasks:**
1. Built-in feature importance (RF, XGBoost)
2. SHAP (SHapley Additive exPlanations) analysis
3. Permutation importance
4. Linear model coefficients (LASSO)
5. Compare and rank features across methods
6. Identify consensus top features
7. Correlation analysis

**Outputs:**
- Multi-method importance rankings
- SHAP visualizations
- Consensus feature list
- Correlation matrices

**Methods:** SHAP, Permutation, LASSO

---

## 1. Import Libraries


In [1]:
import os, sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Lasso
from imblearn.over_sampling import SMOTE

# SHAP for model interpretation
import shap

import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

print('✓ Libraries imported')
print(f'SHAP version: {shap.__version__}')


✓ Libraries imported
SHAP version: 0.49.1


## 2. Project Setup


In [2]:
PROJECT_ROOT = Path('/Users/harryirving/Development/projects/ai-ml/BikeAIv5')
os.chdir(PROJECT_ROOT)

FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'classical'
MODELS_DIR = PROJECT_ROOT / 'models' / 'classical'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = PROJECT_ROOT / 'logs'

print(f'Models: {MODELS_DIR}')
print(f'Results: {RESULTS_DIR}')


Models: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/classical
Results: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


## 3. Setup MLflow


In [3]:
mlflow.set_tracking_uri(f'file://{LOGS_DIR / "mlruns"}')
mlflow.set_experiment('angle_grinder_pipeline')
print('✓ MLflow configured')


✓ MLflow configured


## 4. Load Data and Models


In [4]:
# Select feature set
FEATURE_SET = 'mfcc'  # Change to 'gtcc' or 'combined' as needed

# Load features
if FEATURE_SET == 'mfcc':
    X = np.load(FEATURES_DIR / 'mfcc_unbalanced_features.npy')
elif FEATURE_SET == 'gtcc':
    X = np.load(FEATURES_DIR / 'gtcc_unbalanced_features.npy')
else:
    X = np.load(FEATURES_DIR / 'combined_unbalanced_features.npy')

y = np.load(FEATURES_DIR / 'labels.npy')

print(f'Loaded features: {X.shape}')
print(f'Feature set: {FEATURE_SET}')

# Recreate splits
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42)

# Balance and scale
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

scaler = joblib.load(MODELS_DIR / f'scaler_{FEATURE_SET}.pkl')
X_train_scaled = scaler.transform(X_train_balanced)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'\nData ready: Train={X_train_scaled.shape}, Val={X_val_scaled.shape}, Test={X_test_scaled.shape}')


Loaded features: (60348, 208)
Feature set: mfcc


ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 5. Load Tuned Models


In [ ]:
models = {}
model_files = ['random_forest', 'xgboost', 'svm']

for name in model_files:
    model_path = MODELS_DIR / f'{name}_tuned_{FEATURE_SET}.pkl'
    if model_path.exists():
        models[name] = joblib.load(model_path)
        print(f'✓ Loaded: {name}')

print(f'\nLoaded {len(models)} models')


## 6. Method 1: Built-in Feature Importance

From Random Forest and XGBoost


In [ ]:
importance_methods = {}

# Random Forest
if 'random_forest' in models:
    rf_importance = models['random_forest'].feature_importances_
    importance_methods['RF Built-in'] = rf_importance
    print(f'✓ Extracted RF importance')

# XGBoost
if 'xgboost' in models:
    xgb_importance = models['xgboost'].feature_importances_
    importance_methods['XGB Built-in'] = xgb_importance
    print(f'✓ Extracted XGBoost importance')

print(f'\nBuilt-in methods: {len(importance_methods)}')


## 7. Method 2: SHAP Values

Model-agnostic explanation using SHAP


In [ ]:
# Use a subset of data for SHAP (computational efficiency)
n_shap_samples = min(500, X_test_scaled.shape[0])
X_shap = X_test_scaled[:n_shap_samples]
y_shap = y_test[:n_shap_samples]

print(f'Computing SHAP values on {n_shap_samples} samples...')
print('='*70)

shap_values_dict = {}

# Random Forest SHAP
if 'random_forest' in models:
    print('\nRandom Forest SHAP...')
    explainer_rf = shap.TreeExplainer(models['random_forest'])
    shap_values_rf = explainer_rf.shap_values(X_shap)
    
    # For binary classification, take class 1 (grinder)
    if isinstance(shap_values_rf, list):
        shap_values_rf = shap_values_rf[1]
    
    shap_importance_rf = np.abs(shap_values_rf).mean(axis=0)
    importance_methods['RF SHAP'] = shap_importance_rf
    shap_values_dict['random_forest'] = shap_values_rf
    print(f'  ✓ Shape: {shap_values_rf.shape}')

# XGBoost SHAP
if 'xgboost' in models:
    print('\nXGBoost SHAP...')
    explainer_xgb = shap.TreeExplainer(models['xgboost'])
    shap_values_xgb = explainer_xgb.shap_values(X_shap)
    shap_importance_xgb = np.abs(shap_values_xgb).mean(axis=0)
    importance_methods['XGB SHAP'] = shap_importance_xgb
    shap_values_dict['xgboost'] = shap_values_xgb
    print(f'  ✓ Shape: {shap_values_xgb.shape}')

# SVM SHAP (using KernelExplainer - slower)
if 'svm' in models and n_shap_samples <= 100:
    print('\nSVM SHAP (this may take a while)...')
    # Use even smaller sample for SVM
    X_svm_shap = X_shap[:100]
    X_background = shap.sample(X_train_scaled, 50)
    explainer_svm = shap.KernelExplainer(models['svm'].predict_proba, X_background)
    shap_values_svm = explainer_svm.shap_values(X_svm_shap)
    
    if isinstance(shap_values_svm, list):
        shap_values_svm = shap_values_svm[1]
    
    shap_importance_svm = np.abs(shap_values_svm).mean(axis=0)
    importance_methods['SVM SHAP'] = shap_importance_svm
    print(f'  ✓ Shape: {shap_values_svm.shape}')

print(f'\n✓ SHAP analysis complete')


## 8. SHAP Visualizations


In [ ]:
if 'random_forest' in shap_values_dict:
    print('Creating SHAP visualizations...')
    
    # Summary plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_dict['random_forest'], X_shap, show=False, max_display=20)
    plt.title('SHAP Summary Plot - Random Forest')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'06b_shap_summary_{FEATURE_SET}.png', dpi=300, bbox_inches='tight')
    print(f'✓ Saved SHAP summary plot')
    plt.show()
    
    # Bar plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_dict['random_forest'], X_shap, plot_type='bar', show=False, max_display=20)
    plt.title('SHAP Feature Importance - Random Forest')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'06b_shap_bar_{FEATURE_SET}.png', dpi=300, bbox_inches='tight')
    print(f'✓ Saved SHAP bar plot')
    plt.show()


## 9. Method 3: Permutation Importance

Model-agnostic feature importance


In [ ]:
print('Computing Permutation Importance...')
print('='*70)

# Use validation set for permutation importance
for name, model in models.items():
    print(f'\n{name.replace("_", " ").title()}...')
    
    perm_importance = permutation_importance(
        model, X_val_scaled, y_val,
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )
    
    importance_methods[f'{name.upper()} Permutation'] = perm_importance.importances_mean
    print(f'  ✓ Mean importance range: [{perm_importance.importances_mean.min():.4f}, {perm_importance.importances_mean.max():.4f}]')

print(f'\n✓ Permutation importance complete')


## 10. Method 4: Linear Model Coefficients (LASSO)

Train LASSO for feature selection


In [ ]:
print('Training LASSO for linear coefficients...')

# Train LASSO
lasso = Lasso(alpha=0.01, random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train_balanced)

# Get absolute coefficients
lasso_importance = np.abs(lasso.coef_)
importance_methods['LASSO'] = lasso_importance

print(f'✓ LASSO trained')
print(f'  Non-zero coefficients: {np.sum(lasso_importance > 0)}')
print(f'  Coefficient range: [{lasso_importance.min():.4f}, {lasso_importance.max():.4f}]')

# Also train Logistic Regression for comparison
logreg = LogisticRegression(penalty='l2', random_state=42, max_iter=1000)
logreg.fit(X_train_scaled, y_train_balanced)
logreg_importance = np.abs(logreg.coef_[0])
importance_methods['LogReg'] = logreg_importance

print(f'✓ Logistic Regression trained')
print(f'  Coefficient range: [{logreg_importance.min():.4f}, {logreg_importance.max():.4f}]')


## 11. Normalize and Compare All Methods


In [ ]:
# Normalize all importance scores to [0, 1]
normalized_importance = {}

for method_name, importance in importance_methods.items():
    # Min-max normalization
    imp_min = importance.min()
    imp_max = importance.max()
    if imp_max > imp_min:
        normalized = (importance - imp_min) / (imp_max - imp_min)
    else:
        normalized = importance
    normalized_importance[method_name] = normalized

# Create DataFrame
df_importance = pd.DataFrame(normalized_importance)
df_importance['Feature_ID'] = range(len(df_importance))

# Calculate average importance across all methods
df_importance['Mean'] = df_importance.drop('Feature_ID', axis=1).mean(axis=1)
df_importance['Std'] = df_importance.drop(['Feature_ID', 'Mean'], axis=1).std(axis=1)

# Sort by mean importance
df_importance_sorted = df_importance.sort_values('Mean', ascending=False).reset_index(drop=True)

print('Top 10 Features (averaged across all methods):')
print('='*70)
print(df_importance_sorted[['Feature_ID', 'Mean', 'Std']].head(10).to_string(index=False))

# Save full importance table
df_importance_sorted.to_csv(RESULTS_DIR / f'06b_multi_method_importance_{FEATURE_SET}.csv', index=False)
print(f'\n✓ Saved: 06b_multi_method_importance_{FEATURE_SET}.csv')


## 12. Correlation Analysis Between Methods


In [ ]:
# Calculate correlation between different importance methods
methods_only = df_importance.drop(['Feature_ID', 'Mean', 'Std'], axis=1)
correlation_matrix = methods_only.corr()

print('\nCorrelation between importance methods:')
print('='*70)
print(correlation_matrix.round(3))

# Visualize correlation
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Between Feature Importance Methods')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'06b_method_correlation_{FEATURE_SET}.png', dpi=300)
print(f'\n✓ Saved correlation heatmap')
plt.show()


## 13. Consensus Top Features

Features that rank high across multiple methods


In [ ]:
# For each method, get top 50 features
top_n = min(50, X.shape[1])

consensus_scores = np.zeros(X.shape[1])

for method_name, importance in normalized_importance.items():
    top_features = np.argsort(importance)[::-1][:top_n]
    consensus_scores[top_features] += 1

# Features that appear in top N across most methods
consensus_ranking = np.argsort(consensus_scores)[::-1]

print(f'Consensus Top 20 Features:')
print('='*70)
print(f'Feature ID | Appears in Top {top_n} of N Methods | Avg Importance')
print('-'*70)

for i in range(min(20, len(consensus_ranking))):
    feat_id = consensus_ranking[i]
    appearances = int(consensus_scores[feat_id])
    avg_imp = df_importance.loc[df_importance['Feature_ID'] == feat_id, 'Mean'].values[0]
    print(f'{feat_id:10} | {appearances:30} | {avg_imp:.4f}')

# Save consensus features
consensus_features = consensus_ranking[:top_n]
np.save(RESULTS_DIR / f'consensus_features_{FEATURE_SET}.npy', consensus_features)
print(f'\n✓ Saved top {top_n} consensus features')


## 14. Visualize Multi-Method Comparison


In [ ]:
# Plot top features across different methods
top_features_to_plot = 15
top_feature_ids = df_importance_sorted['Feature_ID'].head(top_features_to_plot).values

fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data
methods_to_plot = [col for col in df_importance.columns if col not in ['Feature_ID', 'Mean', 'Std']]
x = np.arange(top_features_to_plot)
width = 0.8 / len(methods_to_plot)

for i, method in enumerate(methods_to_plot):
    values = [df_importance.loc[df_importance['Feature_ID'] == fid, method].values[0] for fid in top_feature_ids]
    ax.bar(x + i * width, values, width, label=method, alpha=0.8)

ax.set_xlabel('Feature ID')
ax.set_ylabel('Normalized Importance')
ax.set_title(f'Top {top_features_to_plot} Features - Multi-Method Comparison')
ax.set_xticks(x + width * (len(methods_to_plot) - 1) / 2)
ax.set_xticklabels([f'F{fid}' for fid in top_feature_ids])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'06b_multi_method_comparison_{FEATURE_SET}.png', dpi=300, bbox_inches='tight')
print('✓ Saved multi-method comparison plot')
plt.show()


## 15. Log to MLflow


In [ ]:
with mlflow.start_run(run_name=f'06b_feature_importance_{FEATURE_SET}'):
    # Log parameters
    mlflow.log_param('feature_set', FEATURE_SET)
    mlflow.log_param('num_features', X.shape[1])
    mlflow.log_param('num_methods', len(importance_methods))
    mlflow.log_param('shap_samples', n_shap_samples)
    mlflow.log_param('consensus_features', top_n)
    
    # Log metrics
    mlflow.log_metric('num_importance_methods', len(importance_methods))
    
    # Average correlation between methods
    avg_correlation = correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean()
    mlflow.log_metric('avg_method_correlation', avg_correlation)
    
    # Log tags
    mlflow.set_tags({
        'stage': 'feature_analysis',
        'notebook': '06b',
        'analysis_type': 'multi_method_importance'
    })
    
    # Log artifacts
    mlflow.log_artifact(str(RESULTS_DIR / f'06b_multi_method_importance_{FEATURE_SET}.csv'))
    mlflow.log_artifact(str(RESULTS_DIR / f'consensus_features_{FEATURE_SET}.npy'))
    mlflow.log_artifact(str(FIGURES_DIR / f'06b_method_correlation_{FEATURE_SET}.png'))
    mlflow.log_artifact(str(FIGURES_DIR / f'06b_multi_method_comparison_{FEATURE_SET}.png'))
    
    if 'random_forest' in shap_values_dict:
        mlflow.log_artifact(str(FIGURES_DIR / f'06b_shap_summary_{FEATURE_SET}.png'))
        mlflow.log_artifact(str(FIGURES_DIR / f'06b_shap_bar_{FEATURE_SET}.png'))
    
    print('\n✓ Logged to MLflow')


## 16. Summary and Next Steps


In [ ]:
print('='*70)
print('FEATURE IMPORTANCE ANALYSIS COMPLETE')
print('='*70)
print(f'\nFeature Set: {FEATURE_SET.upper()}')
print(f'Total Features: {X.shape[1]}')
print(f'\nImportance Methods Used:')
for i, method in enumerate(importance_methods.keys(), 1):
    print(f'  {i}. {method}')
print(f'\nKey Findings:')
print(f'  - Top feature: {df_importance_sorted.iloc[0]["Feature_ID"]} (avg importance: {df_importance_sorted.iloc[0]["Mean"]:.4f})')
print(f'  - Consensus features saved: {top_n}')
print(f'  - Average method correlation: {avg_correlation:.3f}')

high_agreement = correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)]
print(f'  - Method agreement: {(high_agreement > 0.7).sum()}/{len(high_agreement)} pairs with r > 0.7')

print(f'\nOutputs saved to:')
print(f'  - {RESULTS_DIR}')
print(f'  - {FIGURES_DIR}')

print(f'\nNext Steps:')
print(f'  → Use consensus features for optimized models')
print(f'  → 07_custom_cnn_training.ipynb (neural networks)')
print(f'  → 08_yamnet_transfer_learning.ipynb (transfer learning)')
print('='*70)
print('\n✓ Notebook 06b Complete!')
